# LOT 2026 - Cognitive modeling meets computational linguistics (Yevgen Matusevych)
## Day 5 practical
# Grounding and mutual exclusivity in a child-data model

This practical works with **CVCL** (Vong, Wang, Orhan & Lake, 2024), a vision-and-language
model trained on video and speech from a single child's head-mounted camera. We compare it
against **CLIP**, which was trained on web-scale data instead.

There are two parts:

1. **Grounding.** Test how well each model connects words to objects, and ask whether CVCL's
   accuracy tracks how often the child heard each word.
2. **Mutual exclusivity.** Test the child-like tendency to attach a new word to a new object,
   and work out what the model is actually doing.

The models are given to you. The part that takes thought is the test you build around them.

## 1. Setup

Run this cell first. It fetches the model code, installs the packages it needs, downloads the
image set, and prepares the checkpoint for loading. The first run takes a few minutes, mostly
because the image archive is about 324 MB.

It is safe to run again. If a Colab runtime resets and your files are gone, re-running picks up
whatever is missing and skips the rest.

In [ ]:
import os, sys, subprocess

def run(command):
    subprocess.run(command, shell=True, check=True)

# Model code: clone the repo and put it on the import path.
if not os.path.isdir("multimodal-baby"):
    run("git clone https://github.com/wkvong/multimodal-baby.git")
repo_path = os.path.abspath("multimodal-baby")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Dependencies: install each only if it is not already importable.
def install_if_missing(module, pip_name):
    try:
        __import__(module)
    except ImportError:
        run(f"pip -q install {pip_name}")

install_if_missing("clip", "git+https://github.com/openai/CLIP.git")
install_if_missing("pytorch_lightning", "pytorch_lightning")
install_if_missing("pycocoevalcap", "pycocoevalcap")

# The spaCy English model is downloaded with spaCy, not pip.
try:
    import en_core_web_sm  # noqa: F401
except ImportError:
    run("python -m spacy download en_core_web_sm")

# Image set: Konkle "Object Categories" (unzips to ./17-objects/).
import urllib.request, zipfile
KONKLE_URL = "http://olivalab.mit.edu/MM/archives/ObjectCategories.zip"
if not os.path.isdir("17-objects"):
    if not os.path.exists("ObjectCategories.zip"):
        print("Downloading Konkle Object Categories (~324 MB)...")
        request = urllib.request.Request(KONKLE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response, open("ObjectCategories.zip", "wb") as out:
            out.write(response.read())
    with zipfile.ZipFile("ObjectCategories.zip") as archive:
        archive.extractall(".")

# The CVCL checkpoint stores whole objects, not just weights, so recent PyTorch needs
# permission to unpickle them. This checkpoint is from the model's authors, so we allow it.
import torch
_load = torch.serialization.load
def _load_trusting(*args, **kwargs):
    kwargs["weights_only"] = False
    return _load(*args, **kwargs)
torch.serialization.load = _load_trusting
torch.load = _load_trusting

print("Setup done. Data:", os.path.isdir("17-objects"), "| repo on path:", repo_path in sys.path)

## 2. Choose a model

`"cvcl"` is the child-data model and the main focus here. `"clip"` is the web-scale model, useful
as a contrast. Pick one, then run the rest of the notebook. To compare, change this and re-run
from here.

In [ ]:
BACKEND = "cvcl"   # "cvcl" or "clip"

## 3. Load the model

Both models are wrapped behind the same two methods: `encode_image(paths)` and
`encode_text(words)`, each returning unit-length feature vectors. Because everything below talks
to the model only through these two methods, the rest of the notebook does not care which model
you picked.

In [ ]:
import numpy as np
import torch
from PIL import Image


def unit_vectors(x):
    """Scale each row to length 1 so we can compare them with a dot product."""
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)


def load_cvcl():
    from multimodal.multimodal_lit import MultiModalLitModel
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, transform = MultiModalLitModel.load_model(model_name="cvcl")
    model = model.to(device).eval()

    class CVCL:
        def encode_image(self, paths):
            batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths]).to(device)
            with torch.no_grad():
                features = model.encode_image(batch)
            return unit_vectors(features.cpu().numpy())

        def encode_text(self, words):
            tokens, lengths = model.tokenize(list(words))
            with torch.no_grad():
                features = model.encode_text(tokens.to(device), lengths)
            return unit_vectors(features.cpu().numpy())

    return CVCL()


def load_clip():
    from transformers import CLIPModel, CLIPProcessor
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    class CLIP:
        def encode_image(self, paths):
            images = [Image.open(p).convert("RGB") for p in paths]
            inputs = processor(images=images, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_image_features(pixel_values=inputs["pixel_values"])
            return unit_vectors(features.cpu().numpy())

        def encode_text(self, words):
            prompts = [f"a photo of a {word}" for word in words]
            inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                features = model.get_text_features(input_ids=inputs["input_ids"],
                                                   attention_mask=inputs["attention_mask"])
            return unit_vectors(features.cpu().numpy())

    return CLIP()


model = load_cvcl() if BACKEND == "cvcl" else load_clip()
print("Loaded:", BACKEND)

## 4. Load the images

The Konkle Objects set has real photographs of single objects on plain backgrounds, one folder
per category. We keep the 64 categories whose words are in CVCL's vocabulary as the **familiar**
set, and set a few of the remaining categories aside as **novel** objects for Part 6.

Each image is shrunk and placed on a white background so the object sits small in the frame, the
way the CVCL paper prepared these images for testing.

In [ ]:
# The 64 categories, with how many times the child heard each word during training (Vong et al.).
WORD_FREQUENCY = {
    "ball": 481, "cat": 416, "train": 235, "socks": 129, "bottle": 110, "camera": 92,
    "pants": 91, "apple": 77, "watch": 73, "balloon": 68, "bucket": 59, "chair": 54,
    "spoon": 53, "crib": 51, "jacket": 49, "juice": 48, "bowl": 46, "tree": 46,
    "backpack": 44, "bed": 43, "bird": 36, "button": 34, "shoe": 34, "dog": 31, "hat": 28,
    "pen": 27, "leaves": 26, "bike": 23, "butterfly": 23, "cake": 22, "guitar": 21,
    "basket": 19, "umbrella": 18, "phone": 17, "knife": 16, "bagel": 11, "bench": 10,
    "cheese": 10, "clock": 10, "key": 10, "hairbrush": 9, "rock": 9, "turtle": 9,
    "airplane": 8, "ring": 7, "sofa": 7, "broom": 6, "stool": 6, "bell": 5, "cookie": 5,
    "microwave": 5, "scissors": 5, "stamp": 5, "tv": 5, "coin": 4, "necklace": 4,
    "sandwich": 4, "toothpaste": 4, "desk": 3, "fan": 3, "kayak": 3, "pipe": 3,
    "pizza": 3, "tricycle": 3,
}

DATA_DIR = "17-objects"
PREPARED_DIR = "prepared_images"


def prepare_image(source, target, size=224, object_scale=0.5):
    """Shrink the object and centre it on a white square, then save it."""
    image = Image.open(source).convert("RGB")
    factor = object_scale * size / max(image.size)
    small = image.resize((max(1, round(image.width * factor)), max(1, round(image.height * factor))))
    canvas = Image.new("RGB", (size, size), "white")
    canvas.paste(small, ((size - small.width) // 2, (size - small.height) // 2))
    canvas.save(target)


def photos_in(category, limit):
    """Up to `limit` real photo paths from one category folder, ignoring junk and the TestItems subfolder."""
    folder = os.path.join(DATA_DIR, category)
    names = sorted(n for n in os.listdir(folder)
                   if n.lower().endswith((".jpg", ".jpeg", ".png"))
                   and os.path.isfile(os.path.join(folder, n)))
    return [os.path.join(folder, n) for n in names[:limit]]


def load_images(exemplars=8, n_novel=8):
    os.makedirs(PREPARED_DIR, exist_ok=True)
    folders = {d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))}

    def prepare_category(category, tag):
        prepared = []
        for i, source in enumerate(photos_in(category, exemplars)):
            target = os.path.join(PREPARED_DIR, f"{tag}{category}_{i}.jpg")
            prepare_image(source, target)
            prepared.append(target)
        return prepared

    familiar = {c: prepare_category(c, "") for c in WORD_FREQUENCY if c in folders}
    leftover = sorted(folders - set(WORD_FREQUENCY))[:n_novel]
    novel = {c: prepare_category(c, "novel_") for c in leftover}
    return familiar, novel


familiar_images, novel_images = load_images()
print(f"Familiar categories: {len(familiar_images)} of 64")
print("Novel categories:", list(novel_images))

## 5. Grounding: does the model know which object a word names?

For each photo we build a few **four-way trials**: the target image next to three foils from
other categories. The model picks whichever image its text embedding is most similar to. With
four choices, chance is **25%**.

A word on the overall number. The CVCL paper reports high grounding accuracy using the authors'
full evaluation pipeline. We use the same model but our own trials and foils, so the absolute
percentage will not match theirs, and for CVCL it comes out well below CLIP's. That is expected.
The signal worth reading here is the *pattern* across categories, not the single headline number:
which words the model grounds well, which it does not, and whether that lines up with the child's
experience.

That last point is the developmental question. The categories vary enormously in how often the
child heard the word, from *ball* (481 times) down to *pizza* and *tricycle* (3 times each). Does
the model ground the words it heard often better than the rare ones?

In [ ]:
def make_trials(catalog, trials_per_photo=5, foils=3, seed=0):
    """Build four-way trials: a target photo and its word, plus foil photos from other categories."""
    rng = np.random.default_rng(seed)
    categories = list(catalog)
    trials = []
    for category in categories:
        others = [c for c in categories if c != category]
        for target in catalog[category]:
            for _ in range(trials_per_photo):
                foil_categories = rng.choice(others, size=foils, replace=False)
                images = [target] + [rng.choice(catalog[c]) for c in foil_categories]
                rng.shuffle(images)
                trials.append({"word": category, "images": images, "answer": images.index(target)})
    return trials


def evaluate_grounding(model, trials):
    """Score every trial, return overall accuracy and accuracy per category."""
    all_paths = sorted({p for trial in trials for p in trial["images"]})
    image_vec = dict(zip(all_paths, model.encode_image(all_paths)))
    words = sorted({trial["word"] for trial in trials})
    word_vec = dict(zip(words, model.encode_text(words)))

    correct = {w: 0 for w in words}
    total = {w: 0 for w in words}
    for trial in trials:
        similarities = np.stack([image_vec[p] for p in trial["images"]]) @ word_vec[trial["word"]]
        chosen = int(np.argmax(similarities))
        correct[trial["word"]] += int(chosen == trial["answer"])
        total[trial["word"]] += 1

    overall = sum(correct.values()) / sum(total.values())
    by_category = {w: correct[w] / total[w] for w in words}
    return overall, by_category


trials = make_trials(familiar_images)
overall, by_category = evaluate_grounding(model, trials)
print(f"{len(trials)} four-way trials. Overall accuracy: {overall:.1%}  (chance 25%)")

ranked = sorted(by_category.items(), key=lambda item: item[1])
print("Weakest:", [(c, f"{a:.0%}") for c, a in ranked[:5]])
print("Strongest:", [(c, f"{a:.0%}") for c, a in ranked[-5:]])

### Accuracy against word frequency

We sort the categories into frequency bands and average grounding accuracy in each. A rising bar
chart would say the model learned the words it heard often better than the ones it rarely heard.

In [ ]:
import matplotlib.pyplot as plt

BANDS = [("3-5", 3, 5), ("6-20", 6, 20), ("21-80", 21, 80), ("81+", 81, 10000)]

labels, scores = [], []
for name, low, high in BANDS:
    members = [c for c in by_category if low <= WORD_FREQUENCY[c] <= high]
    if members:
        labels.append(name)
        scores.append(np.mean([by_category[c] for c in members]))

plt.figure(figsize=(5.5, 3.3))
plt.bar(labels, scores, color="#3a6ea5")
plt.axhline(0.25, linestyle="--", color="gray", label="chance")
plt.ylim(0, 1)
plt.xlabel("times the word was heard during training")
plt.ylabel("grounding accuracy")
plt.title(f"Grounding vs. word frequency ({BACKEND})")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Mutual exclusivity

Children given a familiar object, a novel object, and a word they have never heard tend to pick
the novel object: they already have a name for the familiar thing, so the new word must belong to
the new thing (Markman & Wachtel, 1988).

We give the model the same choice and count how often it picks the novel object. With two objects,
chance is **50%**.

In [ ]:
def mutual_exclusivity(model, familiar_paths, novel_paths, new_words):
    """Show a familiar and a novel object with a made-up word. How often is the novel object chosen?"""
    familiar_vec = model.encode_image(familiar_paths)
    novel_vec = model.encode_image(novel_paths)
    word_vec = model.encode_text(new_words)

    picked_novel = []
    for familiar in familiar_vec:
        for novel in novel_vec:
            for word in word_vec:
                picked_novel.append((novel @ word) > (familiar @ word))
    return np.mean(picked_novel), len(picked_novel)


familiar_set = [photos[0] for photos in list(familiar_images.values())[:8]]
novel_set = [photos[0] for photos in novel_images.values()]
made_up_words = ["dax", "blicket", "fep", "wug"]

rate, n = mutual_exclusivity(model, familiar_set, novel_set, made_up_words)
print(f"Novel object chosen {rate:.0%} of the time  ({n} trials, chance 50%)")

### Is the novel object really novel to the model?

Mutual exclusivity assumes the new object has no name yet. A model that has seen a lot may already
recognise it. Here we ask the model to name each held-out object using its familiar vocabulary,
and report how confident it is.

In [ ]:
def name_objects(model, paths, vocabulary):
    image_vec = model.encode_image(paths)
    vocab_vec = model.encode_text(vocabulary)
    for path, image in zip(paths, image_vec):
        similarities = image @ vocab_vec.T
        probabilities = np.exp(similarities - similarities.max())
        probabilities /= probabilities.sum()
        best = int(np.argmax(probabilities))
        print(f"{os.path.basename(path):24s} -> {vocabulary[best]:12s} ({probabilities[best]:.0%})")


name_objects(model, novel_set, list(familiar_images))

### Making sense of the result

If the model sits near chance, the natural conclusion is "it lacks the mutual-exclusivity bias."
But before settling on that, it is worth checking whether the test is even well posed for this
model, because for CVCL it is not, and seeing why is the real lesson here.

Try tokenising a couple of the made-up words:

```python
print(cvcl.tokenize(["dax"]))
print(cvcl.tokenize(["blicket"]))
```

CVCL has a fixed, word-level vocabulary built from the speech the child heard. A word it never
saw has no entry, so the tokeniser maps every unknown word to the same *unknown* token. That means
"dax" and "blicket" produce the **same** representation: encoding a novel word here is really just
encoding "unknown," identical for every made-up word. So our forced-choice test isn't comparing
different novel words at all, it is asking where a single unknown-word vector lands, four times
over. The model has no representation of *dax* to reason with.

That is exactly why this version of the test cannot show mutual exclusivity, even in principle. A
child who hears "dax" forms a new word representation on the spot, shaped by the scene. CVCL
cannot, because its vocabulary was fixed at training time. This is also why Vong and Lake (2022)
gave the novel word a representation through a single gradient update before testing: that step is
the minimum needed to make the novel word a distinct, situation-shaped thing
rather than a generic "unknown." And even with that step, they found the bias fragile.

One more check worth running: the "novel" object may not be novel to the model either. If the
naming probe above is confident about it, the premise of the test is doubly broken.

So the reading of Part 6 should not be "CVCL fails mutual exclusivity," but something like "a closed-vocabulary
model can't be asked the mutual-exclusivity question without first being given a way to represent
the new word, which is the whole problem the cognitive phenomenon is about." For contrast, recall
from the lecture that visually grounded *speech* models show a robust bias on this task, because a
spoken novel word is a real signal the model can encode the first time it hears it.